# Multi-label Palm Oil Grievance Classification: Fine-tuning TwHIN-BERT

## Overview

This notebook fine-tunes **TwHIN-BERT** for **multi-label** topic classification of palm-oil grievance texts. Each grievance can belong to one *or more* of 6 topics (in contrast to the single-label version). We train on `dominant_topic_results.csv` (80/20 train/val split) and evaluate on a separate held-out test set from `classified_grievances_multilabel.csv`.

| Label | Topic |
|-------|-------|
| 0 | Topic_0 |
| 1 | Topic_1 |
| 2 | Topic_2 |
| 3 | Topic_3 |
| 4 | Topic_4 |
| 5 | Topic_5 |

## Pipeline

1. Install / import dependencies.
2. Load train and test CSVs.
3. **Duplicate & leakage audit** — check for duplicated texts within each split and any cross-set leakage between train/val/test.
4. 80/20 train/val split. **Multi-label stratified** by default (via `iterative-stratification`) so every topic is proportionally represented in both sides.
5. Tokenise with the TwHIN-BERT tokenizer.
6. **5-fold cross-validation on the training set** to pick the learning rate, ranked by out-of-fold weighted average precision (threshold-free).
7. **Reuse the winning LR out-of-fold predictions** to pick the sigmoid threshold — one CV pass yields both decisions; nothing is judged at an arbitrary 0.5.
8. Retrain the final model on the full training set with the best LR, and save the best checkpoint.
9. Evaluate on the held-out test set with the chosen threshold; per-topic accuracy / F1 / macro / micro.
10. Scatter-plot the per-example probabilities with TP / TN / FP / FN colour coding.

## What was fixed in this rewrite (July 9, 2026)

* **Checkpoint save vs load path mismatch** — the previous cells saved with an underscore and loaded with a space, so reloading always failed on a fresh run.
* **`select(range(309))`** hardcoded the train size to 309 rows. Removed — we now use the full split.
* **`val_loss += loss_fct(...)`** was accumulating tensors and holding the autograd graph across epochs. Switched to `.item()`.
* **`hidden_dropout_prob=0.4`** (4x BERT's default) reset to `0.1`.
* **Three inconsistent random seeds** (`random_seed=10`, `set_seed(42)`, `shuffle(seed=1111)`) unified under a single `SEED` constant.
* **Missing gradient clipping** added (`max_norm=1.0`).
* **`num_epochs=50` with a scheduler calibrated to 50** replaced with `num_epochs=15` + early stopping (patience 3). The scheduler now decays over an achievable horizon.
* **`evaluation_threshold=0.1`** — the previous value was extremely low. Replaced with a threshold *chosen from cross-validated OOF predictions*.


### Updates July 16, 2026:
* Change some of the hyperparameters to match the Twitter Bert Paper
* Train size is different due to drop duplicate text summary's in the bert topic step
* Added additional code to save outputs of results and to verify classification performance

### Updates July 21, 2026:
* Added a single `USE_WEIGHTED` toggle (§2) to switch pos_weight on/off across the whole pipeline; the run name / save paths encode the choice.
* Learning rate is now chosen by 5-fold CV (out-of-fold weighted average precision); the winning LR OOF predictions feed the §9 threshold sweep — one CV pass, both decisions.
* Added per-topic TP/FP/FN/TN with sensitivity / specificity / precision / NPV / F1 for the validation and test sets (`per_topic_confusion`).
* Test-set predictions and metrics are now saved alongside the other results, and the trailing analyses were reorganised into numbered sections.
* Added an optional final refit on train+val (§12, `RETRAIN_ON_TRAINVAL`): §11 keeps the honest train-only val report, then the model is retrained on all non-test data for §10's selected epoch count and used for the held-out test set.

# 1. Installations and Imports

In [ ]:
from platform import python_version; print(python_version())

In [ ]:
!pip install -q "transformers==4.46.3"
!pip install -q -U datasets
!pip install -q iterative-stratification

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
#drive.mount("/content/gdrive", force_remount=True)

In [ ]:
import os
import json
from collections import Counter

import numpy as np
import pandas as pd
import torch
from matplotlib import pyplot as plt
from matplotlib.lines import Line2D

from torch.optim import AdamW
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    set_seed,
)
from datasets import Dataset

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, average_precision_score

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit, MultilabelStratifiedKFold

from tqdm.notebook import tqdm

pd.set_option('display.max_colwidth', None)

# Reduce GPU memory fragmentation on Colab
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Single seed used for split, model init, dataloader shuffle, CV folds
SEED = 42
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 2. Configuration

In [ ]:
# --- Model / experiment naming ---------------------------------------------
MODEL_CHECKPOINT = "Twitter/twhin-bert-base"
NUM_LABELS       = 6
MAX_LENGTH       = 512

# --- Loss weighting toggle -------------------------------------------------
# Flip this ONE switch to train WITH per-class pos_weight (up-weight rare
# positive labels in the BCE loss) or WITHOUT it. It propagates everywhere:
# the LR/threshold cross-validation (§8-§9), the final retrain (§10), and the
# run name / save paths (so weighted and non-weighted runs never overwrite
# each other).
USE_WEIGHTED   = True
POS_WEIGHT_CAP = 10.0          # cap so very rare topics don't get a huge weight
WEIGHT_TAG     = "weighted" if USE_WEIGHTED else "non-weighted"

# --- Cross-validation ------------------------------------------------------
N_CV_FOLDS = 5                 # folds used to pick BOTH the LR (§8) and the threshold (§9)

# --- Final model -----------------------------------------------------------
# §11 reports the train-only model on the held-out val set. When this is True,
# §12 then refits on train+val combined (all non-test data) for the epoch count
# §10's early stopping picked, and uses THAT model for the held-out test set.
RETRAIN_ON_TRAINVAL = True

RUN_NAME = f"20260721_Twitter_multilabel_CVLR_{WEIGHT_TAG}"

# --- Google Drive paths ----------------------------------------------------
DRIVE_ROOT     = "/content/gdrive/MyDrive/Group 3: palm oil topic classifier"

TRAIN_CSV      = f"{DRIVE_ROOT}/Results/20260715_BERTopic_results.csv"
TEST_CSV       = f"{DRIVE_ROOT}/Text Classification Models/classified_grievances_multilabel.csv"
MODELS_DIR     = f"{DRIVE_ROOT}/Text Classification Models/Twitter_Saved_Models"
BEST_MODEL_PATH = f"{MODELS_DIR}/{RUN_NAME}_best_model.pt"

TOPIC_COLS = [f"Topic_{i}" for i in range(NUM_LABELS)]

# Toggle multi-label stratified splitting; falls back to plain random split if False.
USE_STRATIFIED_SPLIT = True

os.makedirs(MODELS_DIR, exist_ok=True)

print(f"Run name : {RUN_NAME}")
print(f"Weighting: {'ON  (pos_weight in BCE)' if USE_WEIGHTED else 'OFF (plain BCE)'}")
print(f"CV folds : {N_CV_FOLDS}")
print(f"Refit on train+val for test: {RETRAIN_ON_TRAINVAL}")

# 3. Load Data

`dominant_topic_df` is used for training/validation. `test_df` is a separate held-out corpus. Text columns are named `Text` in the training file and `summary` in the test file — we standardise both to `Text` after loading.

In [ ]:
dominant_topic_df = pd.read_csv(TRAIN_CSV)
test_df           = pd.read_csv(TEST_CSV)

# Standardise column name so downstream code doesn't care which source it came from.
test_df = test_df.rename(columns={"summary": "Text"})[["pk", "Text"] + TOPIC_COLS]

print(f"Train file : {len(dominant_topic_df)} rows | cols: {dominant_topic_df.columns.tolist()}")
print(f"Test file  : {len(test_df)} rows | cols: {test_df.columns.tolist()}")

print("\nPer-topic positive counts in training file:")
print(dominant_topic_df[TOPIC_COLS].sum().astype(int))
print("\nPer-topic positive counts in test file:")
print(test_df[TOPIC_COLS].sum().astype(int))

# Multi-label cardinality (how many topics per document, on average)
print(f"\nAvg labels per doc -- train: {dominant_topic_df[TOPIC_COLS].sum(axis=1).mean():.2f}, "
      f"test: {test_df[TOPIC_COLS].sum(axis=1).mean():.2f}")

In [ ]:
test_df

# 4. Duplicate & Leakage Audit

We check three things:

1. **Within-file duplicates** — same `Text` appearing more than once inside the training file or the test file. Duplicates inside training just double-count some examples; duplicates that also cross the train/val split boundary silently *inflate* validation metrics.
2. **Train / test leakage** — the same document appearing in both the training source and the test source. If this happens, your reported test metric is optimistically biased because the model has already seen those inputs.
3. **Same text with different labels** — a data-quality signal, not always a bug (multi-label allows the same text to have different multi-hot vectors from different annotators), but worth eyeballing.

We de-duplicate the training file (keeping the first occurrence) *before* the train/val split so no split-boundary leakage is possible. Test duplicates are just reported — we don't drop them from the test corpus so the reported metric matches what the pipeline would see in production.

In [ ]:
def _duplicate_rows(df, text_col):
    """Return the subset of `df` whose text_col value appears more than once,
    sorted so all copies of the same text are grouped together."""
    return (
        df[df.duplicated(subset=[text_col], keep=False)]
          .sort_values(text_col)
          .reset_index(drop=True)
    )

def _dup_report(df, text_col, name):
    n = len(df)
    n_unique = df[text_col].nunique()
    n_dupe_rows = n - n_unique
    print(f"[{name}] {n} rows, {n_unique} unique texts, {n_dupe_rows} duplicated rows")

    dup_rows = _duplicate_rows(df, text_col)
    if len(dup_rows):
        # How many distinct texts are involved, and how many times each repeats.
        counts = dup_rows.groupby(text_col).size().sort_values(ascending=False)
        print(f"  {len(counts)} distinct texts are duplicated; top repeat counts:")
        for txt, cnt in counts.head(5).items():
            print(f"    {cnt}x  {str(txt)[:80]}")
        print(f"\n  All {len(dup_rows)} duplicate rows in [{name}] (grouped by text):")
        display(dup_rows)          # <-- full rows, all columns
    return dup_rows

# --- Within-file duplicates (kept as DataFrames for later inspection) -----
train_dupes = _dup_report(dominant_topic_df, "Text", "TRAIN")
test_dupes  = _dup_report(test_df,           "Text", "TEST")

# --- Same text, different labels? -----------------------------------------
def _conflicting_label_rows(df, text_col, label_cols):
    """Rows whose text_col value appears with more than one distinct label vector."""
    dup_texts = df[df.duplicated(subset=[text_col], keep=False)][text_col].unique()
    disagreeing = []
    for t in dup_texts:
        subset = df[df[text_col] == t]
        # if the label rows are not all identical, this text has conflicting labels
        if subset[label_cols].drop_duplicates().shape[0] > 1:
            disagreeing.append(subset)
    return (
        pd.concat(disagreeing, ignore_index=True)
        if disagreeing else df.iloc[0:0].copy()
    )

train_conflicts = _conflicting_label_rows(dominant_topic_df, "Text", TOPIC_COLS)
test_conflicts  = _conflicting_label_rows(test_df,           "Text", TOPIC_COLS)
print(f"\n[TRAIN] duplicated texts with CONFLICTING labels: {train_conflicts['Text'].nunique()} texts / {len(train_conflicts)} rows")
if len(train_conflicts):
    display(train_conflicts)
print(f"[TEST]  duplicated texts with CONFLICTING labels: {test_conflicts['Text'].nunique()} texts / {len(test_conflicts)} rows")
if len(test_conflicts):
    display(test_conflicts)

# --- Train / test leakage --------------------------------------------------
train_texts = set(dominant_topic_df["Text"].astype(str))
test_texts  = set(test_df["Text"].astype(str))
overlap     = train_texts & test_texts
print(f"\n[TRAIN and TEST] texts appearing in BOTH files: {len(overlap)}")
if overlap:
    # Show the full rows on BOTH sides so you can compare labels
    train_overlap_rows = (
        dominant_topic_df[dominant_topic_df["Text"].isin(overlap)]
        .sort_values("Text").reset_index(drop=True)
    )
    test_overlap_rows = (
        test_df[test_df["Text"].isin(overlap)]
        .sort_values("Text").reset_index(drop=True)
    )
    print(f"\n  Overlap as seen in TRAIN ({len(train_overlap_rows)} rows):")
    display(train_overlap_rows)
    print(f"  Overlap as seen in TEST ({len(test_overlap_rows)} rows):")
    display(test_overlap_rows)
    print("\n  These TRAIN rows will be REMOVED below so they do not inflate the test metric.")

# --- Deduplicate the training file BEFORE splitting -----------------------
# (comment these three lines out if you want to inspect first and skip the drop)
before = len(dominant_topic_df)
dominant_topic_df = dominant_topic_df[~dominant_topic_df["Text"].isin(test_texts)]
dominant_topic_df = dominant_topic_df.drop_duplicates(subset=["Text"]).reset_index(drop=True)
after = len(dominant_topic_df)
print(f"\nTraining file cleaned: {before} -> {after} rows ({before - after} removed).")

# 5. Train / Validation Split

With `USE_STRATIFIED_SPLIT = True` (recommended), we use `MultilabelStratifiedShuffleSplit` from the [`iterative-stratification`](https://github.com/trent-b/iterative-stratification) package. Standard `sklearn.train_test_split(stratify=y)` only supports single-label targets and would silently break on multi-label. Iterative stratification aims to balance every label's positive rate across the two splits, which matters when some topics are rare.

With `USE_STRATIFIED_SPLIT = False` we fall back to plain random splitting for reproducibility of the original notebook.

In [ ]:
X_all = dominant_topic_df["Text"].values
y_all = dominant_topic_df[TOPIC_COLS].values.astype(int)

if USE_STRATIFIED_SPLIT:
    splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    (train_idx, val_idx), = splitter.split(X_all, y_all)
else:
    train_idx, val_idx = train_test_split(
        np.arange(len(X_all)), test_size=0.2, random_state=SEED, shuffle=True
    )

train_df = dominant_topic_df.iloc[train_idx].reset_index(drop=True)
val_df   = dominant_topic_df.iloc[val_idx].reset_index(drop=True)

print(f"Split mode: {'multi-label stratified' if USE_STRATIFIED_SPLIT else 'random'}")
print(f"Train: {len(train_df)} rows | Val: {len(val_df)} rows")

per_class = pd.DataFrame({
    "train_pos": train_df[TOPIC_COLS].sum().astype(int),
    "val_pos":   val_df[TOPIC_COLS].sum().astype(int),
})
per_class["train_pos_rate"] = (per_class["train_pos"] / len(train_df)).round(3)
per_class["val_pos_rate"]   = (per_class["val_pos"]   / len(val_df)).round(3)
print("\nPer-topic positive counts and rates:")
print(per_class)

# Final leakage sanity check on the split itself
split_leak = set(train_df["Text"]).intersection(set(val_df["Text"]))
assert not split_leak, f"LEAK: {len(split_leak)} texts appear in both train and val"
print("\nNo train/val text overlap. Good.")

# 6. Tokenization

We wrap tokenisation in a small helper so we can rebuild the datasets identically for grid search, CV folds, and the final retraining. The label column is a length-6 float vector (multi-hot) — `BCEWithLogitsLoss` requires floats, not ints.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def make_tokenized_dataset(texts, labels, pks=None):
    """Build a torch-formatted HF Dataset ready for the DataLoader."""
    frame = {"Text": list(texts), "labels": [list(map(float, row)) for row in labels]}
    if pks is not None:
        frame["pk"] = list(pks)
    ds = Dataset.from_pandas(pd.DataFrame(frame))
    ds = ds.map(
        lambda ex: tokenizer(
            ex["Text"], padding="max_length", truncation=True, max_length=MAX_LENGTH
        ),
        batched=True,
    )
    ds = ds.remove_columns(["Text"])
    ds.set_format("torch")
    return ds

tokenized_train = make_tokenized_dataset(train_df["Text"].values, train_df[TOPIC_COLS].values)
tokenized_val   = make_tokenized_dataset(val_df["Text"].values,   val_df[TOPIC_COLS].values)
tokenized_test  = make_tokenized_dataset(
    test_df["Text"].values, test_df[TOPIC_COLS].values, pks=test_df["pk"].values
)

print(f"train: {len(tokenized_train)} | val: {len(tokenized_val)} | test: {len(tokenized_test)}")
print("Sample train row:")
print(f"  input_ids shape : {tokenized_train[0]['input_ids'].shape}")
print(f"  labels          : {tokenized_train[0]['labels']}")

In [ ]:
# Per-topic pos_weight for BCE, controlled by the USE_WEIGHTED toggle (§2).
# pos_weight[k] = (# negatives) / (# positives) for topic k, capped at POS_WEIGHT_CAP.
# When USE_WEIGHTED is False, pos_weight is None -> train_and_evaluate uses plain BCE.
y = train_df[TOPIC_COLS].values.astype(int)
n_pos = y.sum(axis=0)
n_neg = len(y) - n_pos

if USE_WEIGHTED:
    raw = n_neg / np.clip(n_pos, 1, None)
    pos_weight = torch.tensor(np.clip(raw, 1.0, POS_WEIGHT_CAP), dtype=torch.float, device=device)
    print(f"USE_WEIGHTED=True  -> per-class pos_weight (cap={POS_WEIGHT_CAP}):")
    for i, w in enumerate(pos_weight.tolist()):
        print(f"  Topic_{i}: n_pos={int(n_pos[i])}, pos_weight={w:.2f}")
else:
    pos_weight = None
    print("USE_WEIGHTED=False -> plain BCE (no class weighting).")
    for i in range(NUM_LABELS):
        print(f"  Topic_{i}: n_pos={int(n_pos[i])}")

# 7. Training Helper

One reusable function: build a fresh model, train with `BCEWithLogitsLoss` + early stopping, return the sigmoid probabilities on the validation set at the *best* epoch (lowest val loss). Used by:

* §8 (LR grid search) — called once per LR with the train/val split.
* §9 (threshold CV) — called once per fold.
* §10 (final retrain) — called once on the full training set, this time with `save_path` so we get a persisted checkpoint.

Fixed defaults (all set here, not tuned):
* `batch_size = 4` — modest for a 512-token sequence on Colab GPUs.
* `weight_decay = 0.01`, `warmup_ratio = 0.10`, `dropout = 0.1` — standard BERT fine-tuning values from the original paper.
* Gradient clipping at `max_norm = 1.0`.
* Loss is class-agnostic BCE (each topic independently) — no class weights by default, but the function accepts a `pos_weight` tensor if you want per-class up-weighting later.

### Notes:

- Updated `weight_decay=0.0` to match the TwitterBert parameters as described in their paper


In [ ]:
BATCH_SIZE            = 4
WEIGHT_DECAY          = 0.0
WARMUP_RATIO          = 0
DROPOUT               = 0.1
GRAD_CLIP             = 1.0
DEFAULT_NUM_EPOCHS    = 25
DEFAULT_ES_PATIENCE   = 3


def build_model():
    """Fresh multi-label classifier head on TwHIN-BERT."""
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT,
        num_labels=NUM_LABELS,
        problem_type="multi_label_classification",
        hidden_dropout_prob=DROPOUT,
        attention_probs_dropout_prob=DROPOUT,
    )
    return model.to(device)


@torch.no_grad()
def predict_probs(model, dataloader):
    """Return (probs, labels) as float32 numpy arrays, in dataloader order."""
    model.eval()
    all_probs, all_labels = [], []
    for batch in dataloader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        logits = model(input_ids, attention_mask=attention_mask).logits
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(batch["labels"].numpy())
    return np.vstack(all_probs), np.vstack(all_labels)


def train_and_evaluate(
    train_ds,
    val_ds,
    lr,
    num_epochs=DEFAULT_NUM_EPOCHS,
    early_stopping_patience=DEFAULT_ES_PATIENCE,
    save_path=None,
    pos_weight=None,
    verbose=True,
):
    """One training run. Returns best val loss, val probabilities at the best epoch, and loss curves."""
    set_seed(SEED)

    model = build_model()
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY, eps=1e-8)
    num_steps = len(train_loader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_RATIO * num_steps),
        num_training_steps=num_steps,
    )
    loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_val_loss = float("inf")
    best_epoch    = -1
    best_val_probs = None
    best_val_labels = None
    epochs_since_improvement = 0
    train_losses, val_losses = [], []

    for epoch in range(num_epochs):
        # --- train ---
        model.train()
        epoch_train_losses = []
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device).float()
            logits = model(input_ids, attention_mask=attention_mask).logits
            loss = loss_fct(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            epoch_train_losses.append(loss.item())
        train_losses.append(float(np.mean(epoch_train_losses)))

        # --- val ---
        model.eval()
        val_loss_sum, n_val_batches = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels         = batch["labels"].to(device).float()
                logits = model(input_ids, attention_mask=attention_mask).logits
                val_loss_sum += loss_fct(logits, labels).item()
                n_val_batches += 1
        avg_val_loss = val_loss_sum / max(n_val_batches, 1)
        val_losses.append(avg_val_loss)

        if verbose:
            print(f"epoch {epoch:>2} | train_loss={train_losses[-1]:.4f} | val_loss={avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss  = avg_val_loss
            best_epoch     = epoch
            best_val_probs, best_val_labels = predict_probs(model, val_loader)
            epochs_since_improvement = 0
            if save_path is not None:
                torch.save({
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "val_loss": best_val_loss,
                }, save_path)
        else:
            epochs_since_improvement += 1
            if epochs_since_improvement >= early_stopping_patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch}")
                break

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "best_val_loss":  best_val_loss,
        "best_epoch":     best_epoch,
        "val_probs":      best_val_probs,
        "val_labels":     best_val_labels,
        "train_losses":   train_losses,
        "val_losses":     val_losses,
    }

# 8. Learning-Rate Selection via 5-Fold Cross-Validation

We choose the learning rate with the **same 5-fold CV** we use for the threshold, instead of a single train/val split. Ranking LRs on the ~50-60 row validation set gives one noisy estimate per LR; 5-fold CV on the training set gives each LR an out-of-fold (OOF) score computed over *all* training rows — a much lower-variance estimate. The threshold already gets this treatment (§9); the LR deserves the same.

**One CV pass, two decisions.** Because we rank LRs by *average precision* (threshold-free), a single CV loop yields both the LR and its threshold:

1. For each LR, run 5-fold CV and collect its OOF probabilities.
2. **LR ranking** = weighted average precision on those OOF probs.
3. **Threshold (§9)** = sweep the *winning* LR's OOF probs — already in hand, no re-training.

Nothing is evaluated at an arbitrary 0.5 threshold anymore. The folds are **frozen once** so every LR sees identical splits (a fair comparison). Per-fold `pos_weight` is recomputed from each fold's own training rows when `USE_WEIGHTED=True`, and is `None` otherwise.

**Compute cost:** `len(LR_GRID) x N_CV_FOLDS` training runs (default 4 x 5 = 20). Reduce `LR_GRID` or `N_CV_FOLDS` if this is too slow on Colab.

In [ ]:
LR_GRID = [1e-5, 2e-5, 3e-5, 5e-5]

X_train_texts = train_df["Text"].values
y_train_multi = train_df[TOPIC_COLS].values.astype(int)

# Freeze the folds ONCE so every LR sees identical splits (fair comparison).
if USE_STRATIFIED_SPLIT:
    cv = MultilabelStratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=SEED)
else:
    cv = KFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=SEED)
folds = list(cv.split(X_train_texts, y_train_multi))


def fold_pos_weight(y_fold_train):
    """Per-fold pos_weight from THIS fold's training rows only (None if unweighted)."""
    if not USE_WEIGHTED:
        return None
    npos = y_fold_train.sum(axis=0)
    nneg = len(y_fold_train) - npos
    return torch.tensor(
        np.clip(nneg / np.clip(npos, 1, None), 1.0, POS_WEIGHT_CAP),
        dtype=torch.float, device=device,
    )


lr_oof, lr_scores = {}, []
for lr in LR_GRID:
    print(f"\n=== lr = {lr:g} ===")
    oof = np.zeros_like(y_train_multi, dtype=np.float32)
    for fold, (tr_idx, ho_idx) in enumerate(folds):
        tr_ds = make_tokenized_dataset(X_train_texts[tr_idx], y_train_multi[tr_idx])
        ho_ds = make_tokenized_dataset(X_train_texts[ho_idx], y_train_multi[ho_idx])
        result = train_and_evaluate(
            tr_ds, ho_ds, lr=lr,
            num_epochs=DEFAULT_NUM_EPOCHS,
            early_stopping_patience=DEFAULT_ES_PATIENCE,
            save_path=None, verbose=False,
            pos_weight=fold_pos_weight(y_train_multi[tr_idx]),
        )
        oof[ho_idx] = result["val_probs"]
        print(f"  fold {fold + 1}/{N_CV_FOLDS}: best_epoch={result['best_epoch']} "
              f"val_loss={result['best_val_loss']:.4f}")
    lr_oof[lr] = oof
    ap = average_precision_score(y_train_multi, oof, average="weighted")
    lr_scores.append({"lr": lr, "cv_AP": round(ap, 4)})
    print(f"  -> lr={lr:g}  CV weighted-AP = {ap:.4f}")

lr_df = pd.DataFrame(lr_scores).sort_values("cv_AP", ascending=False).reset_index(drop=True)
BEST_LR   = float(lr_df.iloc[0]["lr"])
oof_probs = lr_oof[BEST_LR]          # winning LR's OOF -> reused by the §9 threshold sweep

print("\n=== LR grid (5-fold CV, weighted average precision) ===")
print(lr_df.to_string(index=False))
print(f"\nBest LR (5-fold CV, AP) = {BEST_LR:g}")

# 9. Threshold Selection (reuses the winning LR's OOF probabilities)

The learning-rate search in §8 already produced clean **out-of-fold (OOF) probabilities** for the winning LR (`oof_probs`) — predictions on held-out folds the model never trained on. We sweep the decision threshold directly on those, with **no additional training**.

Sweeping on OOF predictions (rather than on the small validation set) avoids overfitting the sweep to ~50-60 noisy rows and keeps the validation set untouched for the final §11 report.

We report the best **global** threshold (one for all 6 topics) and, for reference, the best **per-topic** threshold. Per-topic usually scores higher but overfits more on a small dataset; global is the safer default.

In [ ]:
# Threshold selection runs on the winning LR's OOF probs from §8 -- no re-training.
assert "oof_probs" in dir() and oof_probs is not None, \
    "Run §8 first to produce oof_probs for the winning LR."

# --- Sweep GLOBAL threshold on OOF preds -----------------------------------
thresholds = np.round(np.arange(0.05, 0.95 + 1e-9, 0.05), 3)
global_scores = []
for t in thresholds:
    preds = (oof_probs >= t).astype(int)
    global_scores.append({
        "threshold":  float(t),
        "macro_f1":   f1_score(y_train_multi, preds, average="macro", zero_division=0),
        "micro_f1":   f1_score(y_train_multi, preds, average="micro", zero_division=0),
    })
global_df = pd.DataFrame(global_scores)
best_row  = global_df.loc[global_df["macro_f1"].idxmax()]
BEST_THRESHOLD_GLOBAL = float(best_row["threshold"])
print(f"Best GLOBAL threshold = {BEST_THRESHOLD_GLOBAL} "
      f"(OOF macro-F1 = {best_row['macro_f1']:.3f}, micro-F1 = {best_row['micro_f1']:.3f})")

# --- Also compute PER-TOPIC thresholds for reference -----------------------
per_topic_thresholds = []
for k in range(NUM_LABELS):
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        f1 = f1_score(y_train_multi[:, k], (oof_probs[:, k] >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    per_topic_thresholds.append((best_t, best_f1))

BEST_THRESHOLDS_PER_TOPIC = np.array([t for t, _ in per_topic_thresholds], dtype=np.float32)
print("\nPer-topic best thresholds (from OOF):")
for i, (t, f1) in enumerate(per_topic_thresholds):
    print(f"  Topic_{i}: threshold={t}  OOF F1={f1:.3f}")

# --- Plot macro-F1 vs threshold --------------------------------------------
plt.style.use("fivethirtyeight")
plt.figure(figsize=(9, 4))
plt.plot(global_df["threshold"], global_df["macro_f1"], label="macro-F1")
plt.plot(global_df["threshold"], global_df["micro_f1"], label="micro-F1")
plt.axvline(BEST_THRESHOLD_GLOBAL, color="grey", linestyle="--",
            label=f"best global = {BEST_THRESHOLD_GLOBAL}")
plt.xlabel("Sigmoid threshold")
plt.ylabel("F1 on OOF predictions")
plt.title("OOF F1 vs threshold (winning LR, 5-fold CV on train)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import json

THRESHOLDS_PATH = f"{MODELS_DIR}/{RUN_NAME}_thresholds.json"

thresholds_to_save = {
    "global": float(BEST_THRESHOLD_GLOBAL),
    "per_topic": [float(t) for t in BEST_THRESHOLDS_PER_TOPIC],   # numpy -> plain floats for JSON
    "topic_cols": TOPIC_COLS,   # record ordering so you can't misalign later
}
with open(THRESHOLDS_PATH, "w") as f:
    json.dump(thresholds_to_save, f, indent=2)

print(f"Saved thresholds to {THRESHOLDS_PATH}")
print(thresholds_to_save)

# 10. Retrain Final Model

Now that we've chosen `BEST_LR` (from §8) and `BEST_THRESHOLD_GLOBAL` (from §9), retrain **once** on the full training set. This is the model we save to disk and use for downstream test inference.

In [ ]:
final_result = train_and_evaluate(
    tokenized_train, tokenized_val,
    lr=BEST_LR,
    num_epochs=DEFAULT_NUM_EPOCHS,
    early_stopping_patience=DEFAULT_ES_PATIENCE,
    save_path=BEST_MODEL_PATH,
    verbose=True,
    pos_weight=pos_weight
)

print("\nBest epoch    :", final_result["best_epoch"])
print("Best val loss :", final_result["best_val_loss"])
print("Saved to      :", BEST_MODEL_PATH)

In [ ]:
# Loss curves
epochs = list(range(len(final_result["train_losses"])))
plt.figure(figsize=(9, 4))
plt.plot(epochs, final_result["train_losses"], label="train loss")
plt.plot(epochs, final_result["val_losses"],   label="val loss")
plt.xlabel("epoch")
plt.ylabel("BCE loss")
plt.title("Training curves")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print("\nBest epoch    :", final_result["best_epoch"])
print("Best val loss :", final_result["best_val_loss"])
print("Saved to      :", BEST_MODEL_PATH)

# 11. Load Best Checkpoint and Evaluate on the Validation Set

Sanity check that the saved checkpoint loads cleanly, then compute the val metrics using **the CV-selected threshold** from §9.

In [ ]:
# BEST_MODEL_PATH = "/content/gdrive/MyDrive/Group 3: palm oil topic classifier/Text Classification Models/Twitter_Saved_Models/20260721_Twitter_multilabel_weighted_best_model.pt"
# BEST_THRESHOLD_GLOBAL = 0.55

In [ ]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded checkpoint from epoch {checkpoint['epoch']} (val_loss={checkpoint['val_loss']:.4f})")

val_loader = DataLoader(tokenized_val, batch_size=BATCH_SIZE, shuffle=False)
val_probs, val_labels = predict_probs(model, val_loader)
val_preds = (val_probs >= BEST_THRESHOLD_GLOBAL).astype(int)

print(f"\n--- Val metrics @ threshold {BEST_THRESHOLD_GLOBAL} ---")
print(f"Accuracy      : {accuracy_score(val_labels, val_preds):.4f}")
print(f"F1 (macro)    : {f1_score(val_labels, val_preds, average='macro',    zero_division=0):.4f}")
print(f"F1 (micro)    : {f1_score(val_labels, val_preds, average='micro',    zero_division=0):.4f}")
print(f"F1 (weighted) : {f1_score(val_labels, val_preds, average='weighted', zero_division=0):.4f}")
print("\nPer-topic:")
for k in range(NUM_LABELS):
    print(f"  Topic_{k}: acc={accuracy_score(val_labels[:, k], val_preds[:, k]):.3f}  "
          f"F1={f1_score(val_labels[:, k], val_preds[:, k], zero_division=0):.3f}  "
          f"support={int(val_labels[:, k].sum())}")

### Per-topic confusion & rates (validation set)

For each topic we count **TP / FP / FN / TN** at the chosen threshold and summarise them as the usual diagnostic rates: **sensitivity** (recall / TPR), **specificity** (TNR), **precision** (PPV), **NPV**, **F1** and **accuracy**, plus macro- and micro-averaged summaries. `per_topic_confusion` is reused later for the held-out test set so the two are computed identically.

In [ ]:
def per_topic_confusion(y_true, y_prob, thresholds, topic_names=None):
    """Per-topic TP/FP/FN/TN and rates (sensitivity/recall, specificity,
    precision/PPV, NPV, F1, accuracy) with macro/micro summaries.

    thresholds : scalar (same threshold for every topic) or length-K array.
    Returns (per_topic_df, macro_series, micro_dict).
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    K = y_true.shape[1]
    thr = (np.full(K, float(thresholds)) if np.isscalar(thresholds)
           else np.asarray(thresholds, dtype=float))
    topic_names = topic_names or [f"Topic_{k}" for k in range(K)]

    def sd(a, b):
        return a / b if b else 0.0

    rows = []
    for k in range(K):
        preds = (y_prob[:, k] >= thr[k]).astype(int)
        tr    = y_true[:, k]
        tp = int(((tr == 1) & (preds == 1)).sum())
        tn = int(((tr == 0) & (preds == 0)).sum())
        fp = int(((tr == 0) & (preds == 1)).sum())
        fn = int(((tr == 1) & (preds == 0)).sum())
        n  = tp + tn + fp + fn
        sens = sd(tp, tp + fn)      # sensitivity / recall / TPR
        spec = sd(tn, tn + fp)      # specificity / TNR
        prec = sd(tp, tp + fp)      # precision / PPV
        npv  = sd(tn, tn + fn)
        rows.append({
            "topic": topic_names[k], "thr": round(float(thr[k]), 2),
            "TP": tp, "FP": fp, "FN": fn, "TN": tn, "support": tp + fn,
            "TP_rate": round(sd(tp, n), 3), "FP_rate": round(sd(fp, n), 3),
            "FN_rate": round(sd(fn, n), 3), "TN_rate": round(sd(tn, n), 3),
            "sensitivity": round(sens, 3), "specificity": round(spec, 3),
            "precision": round(prec, 3), "npv": round(npv, 3),
            "f1": round(sd(2 * prec * sens, prec + sens), 3),
            "accuracy": round(sd(tp + tn, n), 3),
        })
    df = pd.DataFrame(rows)

    macro = df[["sensitivity", "specificity", "precision", "npv", "f1", "accuracy"]].mean().round(3)
    T = df[["TP", "FP", "FN", "TN"]].sum()
    total = int(T["TP"] + T["TN"] + T["FP"] + T["FN"])
    mp = sd(T["TP"], T["TP"] + T["FP"])
    ms = sd(T["TP"], T["TP"] + T["FN"])
    micro = {
        "sensitivity": round(ms, 3),
        "specificity": round(sd(T["TN"], T["TN"] + T["FP"]), 3),
        "precision":   round(mp, 3),
        "f1":          round(sd(2 * mp * ms, mp + ms), 3),
        "accuracy":    round(sd(T["TP"] + T["TN"], total), 3),
    }
    return df, macro, micro


# --- Validation set: per-topic confusion + rates at the CV-selected threshold ---
val_conf_df, val_macro, val_micro = per_topic_confusion(
    val_labels, val_probs, BEST_THRESHOLD_GLOBAL, topic_names=TOPIC_COLS
)
print(f"Per-topic confusion & rates on the VALIDATION set @ global threshold {BEST_THRESHOLD_GLOBAL}:")
print(val_conf_df.to_string(index=False))
print("\nMacro-averaged:", "  ".join(f"{k}={v}" for k, v in val_macro.items()))
print("Micro-averaged:", "  ".join(f"{k}={v}" for k, v in val_micro.items()))

### Weighted AP Results

`/content/gdrive/MyDrive/Group 3: palm oil topic classifier/Text Classification Models/Twitter_Saved_Models/20260721_Twitter_multilabel_AP_LR_weighted_best_model.pt`

Loaded checkpoint from epoch 9 (val_loss=0.5151)

--- Val metrics @ threshold 0.65 ---
* Accuracy      : 0.7067
* F1 (macro)    : 0.7851
* F1 (micro)    : 0.8136
* F1 (weighted) : 0.8134

Per-topic:
  * Topic_0: acc=0.920  F1=0.875  support=24
  * Topic_1: acc=0.907  F1=0.800  support=16
  * Topic_2: acc=0.893  F1=0.733  support=15
  * Topic_3: acc=0.920  F1=0.800  support=13
  * Topic_4: acc=0.987  F1=0.957  support=12
  * Topic_5: acc=0.933  F1=0.545  support=6

### Weighted Results (F1-macro)

`content/gdrive/MyDrive/Group 3: palm oil topic classifier/Text Classification Models/Twitter_Saved_Models/20260721_Twitter_multilabel_macroF1_weighted_best_model.pt`

Loaded checkpoint from epoch 5 (val_loss=0.5249)

--- Val metrics @ threshold 0.55 ---
* Accuracy      : 0.6400
* F1 (macro)    : 0.7934
* F1 (micro)    : 0.8042
* F1 (weighted) : 0.8110

Per-topic:
  * Topic_0: acc=0.907  F1=0.851  support=24
  * Topic_1: acc=0.947  F1=0.889  support=16
  * Topic_2: acc=0.880  F1=0.727  support=15
  * Topic_3: acc=0.827  F1=0.667  support=13
  * Topic_4: acc=0.987  F1=0.960  support=12
  * Topic_5: acc=0.960  F1=0.667  support=6

### Non-weighted AP Results

`/content/gdrive/MyDrive/Group 3: palm oil topic classifier/Text Classification Models/Twitter_Saved_Models/20260721_Twitter_multilabel_AP_LR_non-weighted_best_model.pt`

Loaded checkpoint from epoch 6 (val_loss=0.1798)

--- Val metrics @ threshold 0.35 ---
* Accuracy      : 0.7467
* F1 (macro)    : 0.8256
* F1 (micro)    : 0.8457
* F1 (weighted) : 0.8460

Per-topic:
  * Topic_0: acc=0.933  F1=0.889  support=24
  * Topic_1: acc=0.960  F1=0.903  support=16
  * Topic_2: acc=0.880  F1=0.710  support=15
  * Topic_3: acc=0.960  F1=0.897  support=13
  * Topic_4: acc=0.960  F1=0.889  support=12
  * Topic_5: acc=0.947  F1=0.667  support=6

### Non-weighted Results (F1-macro) (Seems the same as the Non-weighted AP results)

`/content/gdrive/MyDrive/Group 3: palm oil topic classifier/Text Classification Models/Twitter_Saved_Models/20260721_Twitter_multilabel_AP_LR_non-weighted_best_model.pt`

Loaded checkpoint from epoch 6 (val_loss=0.1798)

--- Val metrics @ threshold 0.35 ---
* Accuracy      : 0.7467
* F1 (macro)    : 0.8256
* F1 (micro)    : 0.8457
* F1 (weighted) : 0.8460

Per-topic:
  * Topic_0: acc=0.933  F1=0.889  support=24
  * Topic_1: acc=0.960  F1=0.903  support=16
  * Topic_2: acc=0.880  F1=0.710  support=15
  * Topic_3: acc=0.960  F1=0.897  support=13
  * Topic_4: acc=0.960  F1=0.889  support=12
  * Topic_5: acc=0.947  F1=0.667  support=6

### Evaluate Predictions on the Validation Set

In [ ]:
# ---- Build a long table: one row per (document, topic) pair ----
val_texts     = val_df["Text"].values
val_doc_nums  = val_df["Document_Num"].values   # <-- pull the real doc IDs

rows = []
for i, (doc_num, text) in enumerate(zip(val_doc_nums, val_texts)):
    for k in range(NUM_LABELS):
        prob = float(val_probs[i, k])
        pred = int(val_preds[i, k])
        true = int(val_labels[i, k])
        if   true == 1 and pred == 1: outcome = "TP"
        elif true == 0 and pred == 0: outcome = "TN"
        elif true == 0 and pred == 1: outcome = "FP"
        else:                          outcome = "FN"
        rows.append({
            "Document_Num": doc_num,
            "topic":        f"Topic_{k}",
            "true":         true,
            "pred":         pred,
            "prob":         round(prob, 3),
            "outcome":      outcome,
            "text":         text,
        })
val_pred_df = pd.DataFrame(rows)

# ---- Summary counts ----
print("Outcome counts across all (document x topic) cells:")
print(val_pred_df["outcome"].value_counts().to_string())

per_topic = (val_pred_df.groupby(["topic", "outcome"]).size()
             .unstack(fill_value=0)
             .reindex(columns=["TP", "TN", "FP", "FN"], fill_value=0))
print("\nPer-topic outcome breakdown:")
print(per_topic)

# ---- All errors, most-confident wrong first ----
errors = val_pred_df[val_pred_df["outcome"].isin(["FP", "FN"])].copy()
errors["error_confidence"] = np.where(
    errors["outcome"] == "FP", errors["prob"], 1 - errors["prob"]
)
errors = errors.sort_values("error_confidence", ascending=False).reset_index(drop=True)

print(f"\n{len(errors)} incorrect predictions "
      f"(FP={int((errors['outcome']=='FP').sum())}, "
      f"FN={int((errors['outcome']=='FN').sum())})")
print("Top 30 most-confident mistakes (these are the most diagnostic):")
display(errors[["Document_Num", "topic", "true", "pred", "prob", "outcome", "text"]].head(30))

# ---- Per-document view: which docs are the model struggling with? ----
doc_text_map = dict(zip(val_doc_nums, val_texts))

per_doc = (val_pred_df.assign(is_err=val_pred_df["outcome"].isin(["FP", "FN"]))
           .groupby("Document_Num")
           .agg(n_errors=("is_err", "sum"),
                wrong_topics=("topic", lambda s: ", ".join(
                    val_pred_df.loc[s.index]
                    .query("outcome in ['FP','FN']")["topic"].tolist()
                ))))
per_doc["text"] = [doc_text_map[dn] for dn in per_doc.index]
per_doc = per_doc.sort_values("n_errors", ascending=False)

print(f"\nDocuments with 0 errors (all 6 topics correct): "
      f"{(per_doc['n_errors'] == 0).sum()} / {len(per_doc)}")
print("Distribution of errors per document:")
print(per_doc["n_errors"].value_counts().sort_index().to_string())

print("\nWorst documents (most topics wrong):")
display(per_doc[per_doc["n_errors"] > 0].head(15)[["n_errors", "wrong_topics", "text"]])

print(f"\nTip: `errors` has {len(errors)} rows and `per_doc` has {len(per_doc)} rows — "
      f"you can .to_csv() either to Drive for closer inspection.")

In [ ]:
import json
import numpy as np

THRESHOLDS_PATH = f"{MODELS_DIR}/{RUN_NAME}_thresholds.json"
with open(THRESHOLDS_PATH) as f:
    loaded = json.load(f)

# Sanity check the topic ordering matches this session's TOPIC_COLS
assert loaded["topic_cols"] == TOPIC_COLS, (
    f"Topic ordering mismatch!\n saved: {loaded['topic_cols']}\n now:   {TOPIC_COLS}"
)

BEST_THRESHOLD_GLOBAL     = float(loaded["global"])
BEST_THRESHOLDS_PER_TOPIC = np.array(loaded["per_topic"], dtype=np.float32)

print("Loaded thresholds:")
print(f"  global: {BEST_THRESHOLD_GLOBAL}")
for i, t in enumerate(BEST_THRESHOLDS_PER_TOPIC):
    print(f"  Topic_{i}: {t}")

## Save Results



In [ ]:
# ==== Save §8–§11 results ====
import json
from pathlib import Path

RESULTS_DIR = Path(f"{MODELS_DIR}/{RUN_NAME}_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- 1. summary.json: all scalars, hyperparameters, and final metrics ---
summary = {
    "run_name":              RUN_NAME,
    "model_checkpoint":      MODEL_CHECKPOINT,
    "seed":                  SEED,
    "num_labels":            NUM_LABELS,
    "max_length":            MAX_LENGTH,
    "use_stratified_split":  USE_STRATIFIED_SPLIT,
    "n_train": int(len(train_df)),
    "n_val":   int(len(val_df)),
    "n_test":  int(len(test_df)),

    # Fixed hyperparameters from §7
    "hyperparameters": {
        "batch_size":           BATCH_SIZE,
        "weight_decay":         WEIGHT_DECAY,
        "warmup_ratio":         WARMUP_RATIO,
        "dropout":              DROPOUT,
        "grad_clip":            GRAD_CLIP,
        "default_num_epochs":   DEFAULT_NUM_EPOCHS,
        "default_es_patience":  DEFAULT_ES_PATIENCE,
    },

    # Selected in §8/§9
    "selected": {
        "best_lr":                    BEST_LR,
        "best_threshold_global":      BEST_THRESHOLD_GLOBAL,
        "best_thresholds_per_topic":  BEST_THRESHOLDS_PER_TOPIC.tolist(),
    },

    # Final retrain (§10)
    "final_training": {
        "best_epoch":    int(final_result["best_epoch"]),
        "best_val_loss": float(final_result["best_val_loss"]),
        "train_losses":  [float(x) for x in final_result["train_losses"]],
        "val_losses":    [float(x) for x in final_result["val_losses"]],
    },

    # Val metrics at chosen threshold (§11)
    "val_metrics_at_best_threshold": {
        "threshold":   BEST_THRESHOLD_GLOBAL,
        "accuracy":    float(accuracy_score(val_labels, val_preds)),
        "f1_macro":    float(f1_score(val_labels, val_preds, average="macro",    zero_division=0)),
        "f1_micro":    float(f1_score(val_labels, val_preds, average="micro",    zero_division=0)),
        "f1_weighted": float(f1_score(val_labels, val_preds, average="weighted", zero_division=0)),
        "macro_rates": {k: float(v) for k, v in val_macro.items()},
        "micro_rates": {k: float(v) for k, v in val_micro.items()},
        "per_topic": [
            {
                "topic":    f"Topic_{k}",
                "accuracy": float(accuracy_score(val_labels[:, k], val_preds[:, k])),
                "f1":       float(f1_score(val_labels[:, k], val_preds[:, k], zero_division=0)),
                "support":  int(val_labels[:, k].sum()),
            }
            for k in range(NUM_LABELS)
        ],
    },

    "artifacts": {
        "best_model_checkpoint": BEST_MODEL_PATH,
    },
}
with open(RESULTS_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# --- 2. §8: LR cross-validation results (weighted AP per LR) ---
lr_df.to_csv(RESULTS_DIR / "lr_cv_results.csv", index=False)

# --- 3. §9: threshold sweep (global) ---
global_df.to_csv(RESULTS_DIR / "threshold_sweep_global.csv", index=False)

# --- 4. §9: per-topic threshold sweep (rebuild for saving) ---
per_topic_rows = []
for k in range(NUM_LABELS):
    for t in thresholds:
        preds = (oof_probs[:, k] >= t).astype(int)
        per_topic_rows.append({
            "topic":     f"Topic_{k}",
            "threshold": float(t),
            "f1":        f1_score(y_train_multi[:, k], preds, zero_division=0),
        })
pd.DataFrame(per_topic_rows).to_csv(RESULTS_DIR / "threshold_sweep_per_topic.csv", index=False)

# --- 5. §9: OOF predictions with true labels ---
oof_out = pd.DataFrame({"Text": X_train_texts})
for k in range(NUM_LABELS):
    oof_out[f"prob_Topic_{k}"] = oof_probs[:, k]
    oof_out[f"true_Topic_{k}"] = y_train_multi[:, k]
oof_out.to_csv(RESULTS_DIR / "cv_oof_predictions.csv", index=False)

# --- 6. §10: training curves ---
curves_df = pd.DataFrame({
    "epoch":      range(len(final_result["train_losses"])),
    "train_loss": final_result["train_losses"],
    "val_loss":   final_result["val_losses"],
})
curves_df.to_csv(RESULTS_DIR / "training_curves.csv", index=False)

# --- 7. §11: val predictions (probabilities + hard preds + truth + text) ---
val_out = pd.DataFrame({"Text": val_df["Text"].values})
for k in range(NUM_LABELS):
    val_out[f"prob_Topic_{k}"] = val_probs[:, k]
    val_out[f"pred_Topic_{k}"] = val_preds[:, k]
    val_out[f"true_Topic_{k}"] = val_labels[:, k]
val_out.to_csv(RESULTS_DIR / "val_predictions.csv", index=False)

# --- 7b. §11: per-topic confusion & rates on the validation set ---
val_conf_df.to_csv(RESULTS_DIR / "val_confusion_by_topic.csv", index=False)

# --- 8. Save the train/val split itself for reproducibility ---
train_df.to_csv(RESULTS_DIR / "train_split.csv", index=False)
val_df.to_csv(  RESULTS_DIR / "val_split.csv",   index=False)

# --- Index of what was written ---
print(f"All §8–§11 results saved to:\n  {RESULTS_DIR}\n")
for p in sorted(RESULTS_DIR.iterdir()):
    print(f"  {p.name:<40}  {p.stat().st_size:>10,} bytes")

In [ ]:
!pip freeze > "/content/gdrive/MyDrive/Group 3: palm oil topic classifier/Text Classification Models/Twitter/requirements.txt"

# 12. Retrain the Final Model on Train + Validation

§11 reported the **train-only** model on a genuinely held-out validation set — that stays our honest val estimate. For the model we actually apply to the test set (and would deploy), we now **refit on all non-test labeled data (train + val)**, reusing the LR (§8) and threshold (§9) already selected. Adding val to the fit changes only the model, not any selection decision, so there is no leakage, and the test file is still fully held out.

Because train+val leaves **no held-out set to early-stop on**, we train for a **fixed number of epochs = the best epoch that val-based early stopping picked in §10** (`final_result["best_epoch"] + 1`). `pos_weight` is recomputed from the combined rows. The result is saved to a **separate checkpoint** so the §10 train-only model that §11 reported on is not clobbered.

Set `RETRAIN_ON_TRAINVAL = False` (§2) to skip this and keep using the §10 train-only checkpoint for test.

In [ ]:
# Final model for test: refit on train+val (all non-test data), or keep §10's model.
FULL_MODEL_PATH = f"{MODELS_DIR}/{RUN_NAME}_trainval_best_model.pt"

if RETRAIN_ON_TRAINVAL:
    # Combined train+val (val is a subset of the training CSV; the test file is separate).
    trainval_df = pd.concat([train_df, val_df], ignore_index=True)
    assert not set(trainval_df["Text"]).intersection(set(test_df["Text"])), \
        "LEAK: a train/val text also appears in the test file"

    X_tv = trainval_df["Text"].values
    y_tv = trainval_df[TOPIC_COLS].values.astype(int)
    tokenized_trainval = make_tokenized_dataset(X_tv, y_tv)

    # Recompute pos_weight from the combined rows (or None if unweighted).
    if USE_WEIGHTED:
        npos = y_tv.sum(axis=0)
        nneg = len(y_tv) - npos
        pos_weight_tv = torch.tensor(
            np.clip(nneg / np.clip(npos, 1, None), 1.0, POS_WEIGHT_CAP),
            dtype=torch.float, device=device,
        )
    else:
        pos_weight_tv = None

    # Fixed epoch budget = the epoch val-based early stopping selected in §10.
    FINAL_EPOCHS = int(final_result["best_epoch"]) + 1
    print(f"Refitting on {len(trainval_df)} train+val rows for {FINAL_EPOCHS} epochs "
          f"(lr={BEST_LR:g}, weighted={USE_WEIGHTED}).")

    def train_on_all(train_ds, lr, num_epochs, pos_weight=None, save_path=None):
        """Train for a FIXED number of epochs on the full dataset (no val / no early stopping)."""
        set_seed(SEED)
        model = build_model()
        loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY, eps=1e-8)
        num_steps = len(loader) * num_epochs
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(WARMUP_RATIO * num_steps),
            num_training_steps=num_steps,
        )
        loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        for epoch in range(num_epochs):
            model.train()
            epoch_losses = []
            for batch in loader:
                optimizer.zero_grad()
                input_ids      = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels         = batch["labels"].to(device).float()
                logits = model(input_ids, attention_mask=attention_mask).logits
                loss = loss_fct(logits, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
                optimizer.step()
                scheduler.step()
                epoch_losses.append(loss.item())
            print(f"  epoch {epoch:>2} | train_loss={np.mean(epoch_losses):.4f}")
        if save_path is not None:
            torch.save({"epoch": num_epochs - 1, "model_state_dict": model.state_dict()}, save_path)
        return model

    model = train_on_all(tokenized_trainval, lr=BEST_LR, num_epochs=FINAL_EPOCHS,
                         pos_weight=pos_weight_tv, save_path=FULL_MODEL_PATH)
    model.eval()
    print(f"\nSaved train+val model to {FULL_MODEL_PATH}")
    print("The §13 test evaluation below uses THIS model.")
else:
    # Keep the §10 train-only checkpoint already loaded into `model` in §11.
    print("RETRAIN_ON_TRAINVAL=False -> using the §10 train-only checkpoint for the test set.")

# 13. Predict on the Held-out Test Set

Apply the chosen model + threshold to the unseen test corpus. Because we removed any test-file texts from the training file back in §4, this is a genuine held-out evaluation (barring within-test duplicates, which are just repeats of unseen examples).

`model` here is the **train+val** refit from §12 when `RETRAIN_ON_TRAINVAL=True`, otherwise the §10 train-only checkpoint loaded in §11.

In [ ]:
test_loader = DataLoader(tokenized_test, batch_size=BATCH_SIZE, shuffle=False)

model.eval()
test_probs_batches, test_pks_batches = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting on test"):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        logits = model(input_ids, attention_mask=attention_mask).logits
        test_probs_batches.append(torch.sigmoid(logits).cpu().numpy())
        test_pks_batches.append(batch["pk"].cpu().numpy())

test_probs = np.vstack(test_probs_batches)
test_pks   = np.concatenate(test_pks_batches)
test_labels = test_df.set_index("pk").loc[test_pks, TOPIC_COLS].values.astype(int)

# Predictions using the CV-selected global threshold.
test_preds = (test_probs >= BEST_THRESHOLD_GLOBAL).astype(int)

# Build a tidy prediction DataFrame keyed by pk.
prob_cols = [f"Topic_{i}_Prob" for i in range(NUM_LABELS)]
prediction_df = pd.DataFrame(test_probs, columns=prob_cols)
prediction_df["pk"] = test_pks
prediction_df["Text"] = prediction_df["pk"].map(test_df.set_index("pk")["Text"])
prediction_df = prediction_df[["pk", "Text"] + prob_cols]
display(prediction_df.head())

print(f"\n--- Test metrics @ threshold {BEST_THRESHOLD_GLOBAL} ---")
print(f"Accuracy      : {accuracy_score(test_labels, test_preds):.4f}")
print(f"F1 (macro)    : {f1_score(test_labels, test_preds, average='macro',    zero_division=0):.4f}")
print(f"F1 (micro)    : {f1_score(test_labels, test_preds, average='micro',    zero_division=0):.4f}")
print(f"F1 (weighted) : {f1_score(test_labels, test_preds, average='weighted', zero_division=0):.4f}")
print("\nPer-topic (global threshold):")
for k in range(NUM_LABELS):
    print(f"  Topic_{k}: acc={accuracy_score(test_labels[:, k], test_preds[:, k]):.3f}  "
          f"F1={f1_score(test_labels[:, k], test_preds[:, k], zero_division=0):.3f}  "
          f"support={int(test_labels[:, k].sum())}")

# Optional: also report using the per-topic thresholds selected in §9.
test_preds_pt = (test_probs >= BEST_THRESHOLDS_PER_TOPIC[None, :]).astype(int)
print(f"\n--- Test metrics with PER-TOPIC thresholds {BEST_THRESHOLDS_PER_TOPIC.tolist()} ---")
print(f"F1 (macro)    : {f1_score(test_labels, test_preds_pt, average='macro',    zero_division=0):.4f}")
print(f"F1 (micro)    : {f1_score(test_labels, test_preds_pt, average='micro',    zero_division=0):.4f}")

# 14. Test-Set Performance (per-topic confusion & rates)

Same `per_topic_confusion` table as the validation set (§11), now on the held-out test corpus, for **both** threshold strategies (global vs per-topic), followed by a side-by-side comparison of the headline macro/micro numbers.

In [ ]:
# Per-topic confusion & rates on the TEST set, for both threshold strategies.
test_true_mat = test_df.set_index("pk").loc[prediction_df["pk"], TOPIC_COLS].values.astype(int)
test_prob_mat = prediction_df[prob_cols].values

for name, thr in [(f"GLOBAL threshold = {BEST_THRESHOLD_GLOBAL}", BEST_THRESHOLD_GLOBAL),
                  ("PER-TOPIC thresholds", BEST_THRESHOLDS_PER_TOPIC)]:
    df, macro, micro = per_topic_confusion(test_true_mat, test_prob_mat, thr, topic_names=TOPIC_COLS)
    print("=" * 74)
    print(name)
    print("=" * 74)
    print(df.to_string(index=False))
    print("\n  Macro:", "  ".join(f"{k}={v}" for k, v in macro.items()))
    print("  Micro:", "  ".join(f"{k}={v}" for k, v in micro.items()), "\n")

# Side-by-side comparison of the headline numbers (per_topic - global).
g_df, g_macro, g_micro = per_topic_confusion(test_true_mat, test_prob_mat, BEST_THRESHOLD_GLOBAL, TOPIC_COLS)
p_df, p_macro, p_micro = per_topic_confusion(test_true_mat, test_prob_mat, BEST_THRESHOLDS_PER_TOPIC, TOPIC_COLS)
comparison = pd.DataFrame({
    "metric": ["macro_sensitivity", "macro_specificity", "macro_precision", "macro_f1",
               "micro_sensitivity", "micro_precision", "micro_f1"],
    "global": [g_macro["sensitivity"], g_macro["specificity"], g_macro["precision"], g_macro["f1"],
               g_micro["sensitivity"], g_micro["precision"], g_micro["f1"]],
    "per_topic": [p_macro["sensitivity"], p_macro["specificity"], p_macro["precision"], p_macro["f1"],
                  p_micro["sensitivity"], p_micro["precision"], p_micro["f1"]],
})
comparison["delta"] = (comparison["per_topic"] - comparison["global"]).round(3)
print("=" * 74)
print("COMPARISON (per_topic - global)")
print("=" * 74)
print(comparison.to_string(index=False))

# 15. Save Test Predictions & Metrics

Persist the held-out **test predictions** (probabilities, hard predictions at both threshold strategies, and the ground-truth labels) plus the per-topic confusion tables and headline metrics into the same `RESULTS_DIR` as the §8-§11 artifacts.

In [ ]:
# ==== Save test-set predictions & metrics (§13-§14) into RESULTS_DIR ====
import json

# Full per-document table: probabilities + hard preds (both strategies) + truth.
test_preds_pt = (test_probs >= BEST_THRESHOLDS_PER_TOPIC[None, :]).astype(int)
test_out = pd.DataFrame({"pk": test_pks})
test_out["Text"] = test_out["pk"].map(test_df.set_index("pk")["Text"])
for k in range(NUM_LABELS):
    test_out[f"prob_Topic_{k}"]          = test_probs[:, k]
    test_out[f"pred_global_Topic_{k}"]   = test_preds[:, k]
    test_out[f"pred_pertopic_Topic_{k}"] = test_preds_pt[:, k]
    test_out[f"true_Topic_{k}"]          = test_labels[:, k]
test_out.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)

# Per-topic confusion & rates for both threshold strategies.
test_conf_global, test_macro_g,  test_micro_g  = per_topic_confusion(
    test_labels, test_probs, BEST_THRESHOLD_GLOBAL, topic_names=TOPIC_COLS)
test_conf_pt,     test_macro_pt, test_micro_pt = per_topic_confusion(
    test_labels, test_probs, BEST_THRESHOLDS_PER_TOPIC, topic_names=TOPIC_COLS)
test_conf_global.to_csv(RESULTS_DIR / "test_confusion_global.csv", index=False)
test_conf_pt.to_csv(RESULTS_DIR / "test_confusion_per_topic.csv", index=False)

# Headline test metrics -> json.
test_metrics = {
    "run_name":             RUN_NAME,
    "use_weighted":         USE_WEIGHTED,
    "retrain_on_trainval":  RETRAIN_ON_TRAINVAL,
    "model_checkpoint":     FULL_MODEL_PATH if RETRAIN_ON_TRAINVAL else BEST_MODEL_PATH,
    "n_test":               int(len(test_labels)),
    "threshold_global":     BEST_THRESHOLD_GLOBAL,
    "thresholds_per_topic": [float(t) for t in BEST_THRESHOLDS_PER_TOPIC],
    "global": {
        "accuracy":    float(accuracy_score(test_labels, test_preds)),
        "f1_macro":    float(f1_score(test_labels, test_preds, average="macro",    zero_division=0)),
        "f1_micro":    float(f1_score(test_labels, test_preds, average="micro",    zero_division=0)),
        "f1_weighted": float(f1_score(test_labels, test_preds, average="weighted", zero_division=0)),
        "macro_rates": {k: float(v) for k, v in test_macro_g.items()},
        "micro_rates": {k: float(v) for k, v in test_micro_g.items()},
    },
    "per_topic": {
        "macro_rates": {k: float(v) for k, v in test_macro_pt.items()},
        "micro_rates": {k: float(v) for k, v in test_micro_pt.items()},
    },
}
with open(RESULTS_DIR / "test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

print(f"Saved test artifacts to:\n  {RESULTS_DIR}\n")
for name in ["test_predictions.csv", "test_confusion_global.csv",
             "test_confusion_per_topic.csv", "test_metrics.json"]:
    p = RESULTS_DIR / name
    print(f"  {name:<32} {p.stat().st_size:>10,} bytes")

# 16. Additional Diagnostics

Supplementary error analysis on the test set: total per-document probability mass, a keyword-grounded look at the false positives, and top-1 / top-2 hit rates.

In [ ]:
prediction_df[['Topic_0_Prob', 'Topic_1_Prob', 'Topic_2_Prob', 'Topic_3_Prob', 'Topic_4_Prob', 'Topic_5_Prob']].sum(axis = 1).sort_values()

In [ ]:
import numpy as np, pandas as pd

# topic -> its BERTopic keyword string (from the training-derived column)
topic_keywords = (dominant_topic_df.dropna(subset=["Topic_Keywords"])
                  .groupby("Dominant_Topic")["Topic_Keywords"].first().to_dict())
pk_to_text = prediction_df.set_index("pk")["Text"].to_dict()

try:
    thresholds = np.asarray(BEST_THRESHOLDS_PER_TOPIC, dtype=float)
except NameError:
    thresholds = np.full(len(prob_cols), BEST_THRESHOLD_GLOBAL, dtype=float)

rows = []
for i, prob_col in enumerate(prob_cols):
    probs  = prediction_df[prob_col].values
    truths = test_df.set_index("pk").loc[prediction_df["pk"], f"Topic_{i}"].values.astype(int)
    t      = thresholds[i]
    preds  = (probs >= t).astype(int)

    kws = [k.strip().lower() for k in str(topic_keywords.get(i, "")).split(",") if k.strip()]

    # BERTopic soft membership for this topic, if the test file carries it
    perc_col = f"Topic_{i}_Perc"
    has_perc = perc_col in test_df.columns
    perc = (test_df.set_index("pk").loc[prediction_df["pk"], perc_col].values
            if has_perc else np.full(len(probs), np.nan))

    for pk, p, tr, pr, pc in zip(prediction_df["pk"].values, probs, truths, preds, perc):
        if not (tr == 0 and pr == 1):     # keep only FPs
            continue
        text = str(pk_to_text.get(pk, "")).lower()
        matched = [k for k in kws if k in text]
        rows.append({
            "pk": pk, "topic": f"Topic_{i}",
            "bert_prob": round(float(p), 3),
            "margin_over_thr": round(float(p - t), 3),
            "bertopic_membership": round(float(pc), 3) if has_perc else None,
            "n_keyword_hits": len(matched),
            "matched_keywords": ", ".join(matched),
            "text": pk_to_text.get(pk, ""),
        })

fp_df = pd.DataFrame(rows).sort_values("bert_prob", ascending=False).reset_index(drop=True)
print(f"{len(fp_df)} false positives total\n")

# --- Signal 1: how confident is BERT on its FPs? ---
print("BERT confidence on FPs:")
print(fp_df["bert_prob"].describe()[["mean", "25%", "50%", "75%", "max"]])

# --- Signal 2: do FP docs contain the topic's own keywords? ---
frac_kw = (fp_df["n_keyword_hits"] > 0).mean()
print(f"\nFraction of FPs whose text contains >=1 of the topic's keywords: {frac_kw:.0%}")

# --- Signal 3: were these near-misses in BERTopic's own soft membership? ---
if fp_df["bertopic_membership"].notna().any():
    print("\nBERTopic membership on FP cells (if high, BERTopic 'almost' assigned it):")
    print(fp_df["bertopic_membership"].describe()[["mean", "50%", "max"]])

fp_df.head(20)

In [ ]:
import numpy as np
import pandas as pd

# True label matrix (docs x topics), aligned to prediction_df's pk order
true_mat = test_df.set_index("pk").loc[prediction_df["pk"], [f"Topic_{i}" for i in range(len(prob_cols))]].values.astype(int)
prob_mat = prediction_df[prob_cols].values                      # (docs x topics)

top1_pred = prob_mat.argmax(axis=1)                             # model's single most-probable topic per doc
n_true    = true_mat.sum(axis=1)                                # how many true topics each doc has

# A "hit" = the model's top topic is one of the document's true topics
hit = np.array([true_mat[d, top1_pred[d]] == 1 for d in range(len(top1_pred))])

# Documents with no hand-labeled topic at all can't be scored — track separately
has_label = n_true > 0
hit_rate_all   = hit.mean()
hit_rate_valid = hit[has_label].mean() if has_label.any() else float("nan")

print(f"Documents: {len(top1_pred)}  |  with >=1 hand label: {has_label.sum()}  |  unlabeled: {(~has_label).sum()}")
print(f"\nTop-1 hit rate (all docs):                {hit_rate_all:.3f}")
print(f"Top-1 hit rate (only docs with a label):  {hit_rate_valid:.3f}")
print(f"  -> In {hit[has_label].sum()}/{has_label.sum()} labeled docs, the model's most-probable "
      f"topic was one the annotators assigned.")

In [ ]:
# --- Top-2 hit: is EITHER of the model's two most-probable topics a true label? ---
top2_pred = np.argsort(-prob_mat, axis=1)[:, :2]
hit_top2 = np.array([true_mat[d, top2_pred[d]].max() == 1 for d in range(len(top2_pred))])
print(f"\nTop-2 hit rate (labeled docs): {hit_top2[has_label].mean():.3f}")

# --- Per-topic: when the model's top pick is Topic_i, how often is it correct? ---
rows = []
for i in range(len(prob_cols)):
    sel = top1_pred == i
    n_sel = sel.sum()
    correct = hit[sel].sum()
    rows.append({
        "top_topic": f"Topic_{i}",
        "times_top_pick": int(n_sel),
        "correct": int(correct),
        "precision_of_top_pick": round(correct / n_sel, 3) if n_sel else 0.0,
    })
print("\nWhen each topic is the model's #1 pick, how often is it a true label:")
print(pd.DataFrame(rows).to_string(index=False))

# 17. Prediction Outcome Scatter

Every point is one (document, topic) pair. Colour = confusion-matrix cell (TP / TN / FP / FN) at the chosen threshold. Useful for eyeballing where the model is under/over-confident.

In [ ]:
COLOR_TP = "#FF9D00"
COLOR_TN = "#00A2F3"
COLOR_FP = "#4B2362"
COLOR_FN = "#CE4763"

points = []
for i, prob_col in enumerate(prob_cols):
    probs = prediction_df[prob_col].values
    truths = test_df.set_index("pk").loc[prediction_df["pk"], f"Topic_{i}"].values.astype(int)
    preds = (probs >= BEST_THRESHOLD_GLOBAL).astype(int)
    for pk, p, t, pr in zip(prediction_df["pk"].values, probs, truths, preds):
        if   t == 1 and pr == 1: c = COLOR_TP
        elif t == 0 and pr == 0: c = COLOR_TN
        elif t == 0 and pr == 1: c = COLOR_FP
        else:                    c = COLOR_FN
        points.append((pk, p, c))

fig, ax = plt.subplots(figsize=(14, 6))
for pk, p, c in points:
    ax.scatter(pk, p, color=c, alpha=0.7, s=20)
ax.axhline(BEST_THRESHOLD_GLOBAL, color="grey", linestyle="dashed",
           linewidth=2, label=f"threshold = {BEST_THRESHOLD_GLOBAL}")
ax.set_title("Predicted probability per (document, topic)")
ax.set_xlabel("Document ID (pk)")
ax.set_ylabel("Predicted probability")
ax.set_ylim(0, 1)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(handles=[
    Line2D([0], [0], marker="o", color="w", label="True Positive",  markerfacecolor=COLOR_TP, markersize=8),
    Line2D([0], [0], marker="o", color="w", label="True Negative",  markerfacecolor=COLOR_TN, markersize=8),
    Line2D([0], [0], marker="o", color="w", label="False Positive", markerfacecolor=COLOR_FP, markersize=8),
    Line2D([0], [0], marker="o", color="w", label="False Negative", markerfacecolor=COLOR_FN, markersize=8),
    Line2D([0], [0], color="grey", linestyle="dashed", label=f"threshold = {BEST_THRESHOLD_GLOBAL}"),
], title="Outcome")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

COLOR_TP = "#009E73"   # green  — correct positive
COLOR_FN = "#D55E00"   # vermillion — missed positive (bad)
COLOR_FP = "#CC79A7"   # purple — false alarm
COLOR_TN = "#BBBBBB"   # grey   — correct negative (de-emphasized; usually the majority)

TOPIC_LABELS = [
    "Failed Compensation\n/ Land Conflicts",
    "Environmental\nImpact",
    "Administrative\nViolations",
    "Deforestation",
    "Labor & Human\nRights",
    "Illegal / Contaminated\nFruit Bunches",
]

# Use per-topic thresholds if you have them, else fall back to the global one
try:
    thresholds = np.asarray(BEST_THRESHOLDS_PER_TOPIC, dtype=float)
except NameError:
    thresholds = np.full(len(prob_cols), BEST_THRESHOLD_GLOBAL, dtype=float)

rng = np.random.default_rng(42)          # reproducible jitter
fig, ax = plt.subplots(figsize=(13, 6.5))

legend_counts = {"TP": 0, "FN": 0, "FP": 0, "TN": 0}

for i, prob_col in enumerate(prob_cols):
    probs  = prediction_df[prob_col].values
    truths = test_df.set_index("pk").loc[prediction_df["pk"], f"Topic_{i}"].values.astype(int)
    t      = thresholds[i]
    preds  = (probs >= t).astype(int)

    for p, tr, pr in zip(probs, truths, preds):
        if   tr == 1 and pr == 1: c, key = COLOR_TP, "TP"
        elif tr == 1 and pr == 0: c, key = COLOR_FN, "FN"
        elif tr == 0 and pr == 1: c, key = COLOR_FP, "FP"
        else:                     c, key = COLOR_TN, "TN"
        x = i + rng.uniform(-0.28, 0.28)                       # horizontal jitter
        # de-emphasize the (usually dominant) true negatives
        ax.scatter(x, p, color=c, s=(14 if key == "TN" else 42),
                   alpha=(0.35 if key == "TN" else 0.9),
                   edgecolor="none" if key == "TN" else "black", linewidth=0.4, zorder=2)
        legend_counts[key] += 1

    # per-topic threshold segment
    ax.plot([i - 0.35, i + 0.35], [t, t], color="black", linestyle="--", linewidth=1.6, zorder=3)
    ax.text(i + 0.37, t, f"{t:.2f}", va="center", ha="left", fontsize=8, color="black")

ax.set_xticks(range(len(prob_cols)))
ax.set_xticklabels(TOPIC_LABELS, fontsize=10)
ax.set_ylabel("Predicted probability", fontweight="bold")
ax.set_ylim(-0.02, 1.02)
ax.set_title("Predicted Probabilities and Decision Thresholds by Topic",
             fontsize=15, fontweight="bold", pad=12)
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, linestyle=":", alpha=0.4)
ax.set_axisbelow(True)

ax.legend(handles=[
    Line2D([0],[0], marker="o", color="w", markerfacecolor=COLOR_TP, markersize=9,
           label=f"True Positive ({legend_counts['TP']})"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor=COLOR_FN, markersize=9,
           label=f"False Negative ({legend_counts['FN']})"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor=COLOR_FP, markersize=9,
           label=f"False Positive ({legend_counts['FP']})"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor=COLOR_TN, markersize=7,
           label=f"True Negative ({legend_counts['TN']})"),
    Line2D([0],[0], color="black", linestyle="--", label="Per-topic threshold"),
], title="Outcome", loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.metrics import f1_score

COLOR_TP = "#009E73"   # correct positive
COLOR_FN = "#D55E00"   # missed positive (bad)
COLOR_FP = "#CC79A7"   # false alarm
COLOR_TN = "#BBBBBB"   # correct negative (de-emphasized)

TOPIC_LABELS = [
    "Failed Compensation / Land Conflicts",
    "Environmental Impact",
    "Administrative Violations",
    "Deforestation",
    "Labor & Human Rights",
    "Illegal / Contaminated Fruit Bunches",
]

# per-topic thresholds if available, else global
try:
    thresholds = np.asarray(BEST_THRESHOLDS_PER_TOPIC, dtype=float)
except NameError:
    thresholds = np.full(len(prob_cols), BEST_THRESHOLD_GLOBAL, dtype=float)

rng = np.random.default_rng(42)
n = len(prob_cols)
n_cols = 3
n_rows = int(np.ceil(n / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 8.5), sharey=True)
axes = axes.flatten()

for i, prob_col in enumerate(prob_cols):
    ax = axes[i]
    probs  = prediction_df[prob_col].values
    truths = test_df.set_index("pk").loc[prediction_df["pk"], f"Topic_{i}"].values.astype(int)
    t      = thresholds[i]
    preds  = (probs >= t).astype(int)

    # split into the two truth columns
    for truth_val, xbase, pos_color, neg_color in [
        (0, 0, COLOR_FP, COLOR_TN),   # negatives: above thr = FP, below = TN
        (1, 1, COLOR_TP, COLOR_FN),   # positives: above thr = TP, below = FN
    ]:
        mask = truths == truth_val
        pv   = probs[mask]
        for p in pv:
            correct_side = (p >= t) == (truth_val == 1)
            c = (pos_color if p >= t else neg_color)
            x = xbase + rng.uniform(-0.18, 0.18)
            ax.scatter(x, p, color=c,
                       s=(16 if (truth_val == 0 and p < t) else 46),   # shrink TNs
                       alpha=(0.4 if (truth_val == 0 and p < t) else 0.9),
                       edgecolor="none" if (truth_val == 0 and p < t) else "black",
                       linewidth=0.4, zorder=2)

    # threshold line spanning both columns
    ax.axhline(t, color="black", linestyle="--", linewidth=1.5, zorder=3)
    ax.text(1.52, t, f"thr={t:.2f}", va="center", ha="left", fontsize=8)

    f1 = f1_score(truths, preds, zero_division=0)
    ax.set_title(f"{TOPIC_LABELS[i]}\nF1 = {f1:.2f}  (n_pos = {int(truths.sum())})",
                 fontsize=10, fontweight="bold")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Truth = 0\n(negative)", "Truth = 1\n(positive)"], fontsize=9)
    ax.set_xlim(-0.5, 1.9)
    ax.set_ylim(-0.02, 1.02)
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.grid(True, linestyle=":", alpha=0.4)
    ax.set_axisbelow(True)
    if i % n_cols == 0:
        ax.set_ylabel("Predicted probability", fontweight="bold")

# hide any unused subplot cells
for j in range(n, len(axes)):
    axes[j].set_visible(False)

fig.legend(handles=[
    Line2D([0],[0], marker="o", color="w", markerfacecolor=COLOR_TP, markersize=9, label="True Positive"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor=COLOR_FN, markersize=9, label="False Negative"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor=COLOR_FP, markersize=9, label="False Positive"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor=COLOR_TN, markersize=7, label="True Negative"),
    Line2D([0],[0], color="black", linestyle="--", label="Decision threshold"),
], loc="lower center", ncol=5, frameon=False, bbox_to_anchor=(0.5, -0.02))

fig.suptitle("Predicted Probability Distributions by Topic (test set)",
             fontsize=16, fontweight="bold", y=0.98)
plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.show()